In [141]:
import numpy as np

In [142]:
def load_data_from_csv(filename):
    valid_rows = []
    invalid_count = 0
    
    with open(filename, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) == 4:  # only get valid lines
                try:
                    row = [float(x) for x in parts]
                    valid_rows.append(row)
                except ValueError:
                    invalid_count += 1  # if there is a non-numeric value
            else:
                invalid_count += 1
    
    # Convert to numpy array
    data = np.array(valid_rows)
    
    # Separate input and output
    X = data[:, 0:2]
    Y = data[:, 2:4]
    
    print("Input shape:", X.shape)   
    print("Output shape:", Y.shape) 
    
    return X, Y

path = r"M:\ANN_ASSIGNMENT\Assignment Code\ce889_dataCollection.csv" # data file raw gathering from the game
X, Y = load_data_from_csv(path)
Input_min = np.min(X, axis=0) # the Min value of this neuron doesn't need to be at the same record as the Min of another neuron
Input_max = np.max(X, axis=0)
Output_min = np.min(Y, axis=0) 
Output_max = np.max(Y, axis=0)
print(Input_min,Input_max,Output_min,Output_max)

Input shape: (101384, 2)
Output shape: (101384, 2)
[-1066.64583419    65.56289592] [1095.51122845 1226.88057718] [-7.01940084 -7.9958983 ] [8.         7.98011366]


In [143]:
def scaleData(data):
    data_min = np.min(data, axis=0)
    data_max = np.max(data, axis=0)
    scaled_data = (data - data_min) / (data_max - data_min)
    return scaled_data, data_min, data_max

scaled_X, X_min, X_max = scaleData(X)
scaled_Y, Y_min, Y_max = scaleData(Y)

In [144]:
def split_data(X, Y, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Shuffle and separate data into train, validation, and test.
    - train_ratio: train data ratio
    - val_ratio: validation data ratio
    - test_ratio: test data ratio
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    
    np.random.seed(seed)
    indices = np.arange(X.shape[0]) 
    np.random.shuffle(indices) # suffle data 
    
    X = X[indices]
    Y = Y[indices]
    
    n_train = int(train_ratio * X.shape[0])
    n_val = int(val_ratio * X.shape[0])
    
    X_train, Y_train = X[:n_train], Y[:n_train]
    X_val, Y_val = X[n_train:n_train+n_val], Y[n_train:n_train+n_val]
    X_test, Y_test = X[n_train+n_val:], Y[n_train+n_val:]
    
    print(f"Total: {X.shape[0]}")
    print(f"Train: {X_train.shape[0]} ")
    print(f"Validation: {X_val.shape[0]} ")
    print(f"Test: {X_test.shape[0]} ")
    
    return (X_train, Y_train), (X_val, Y_val), (X_test, Y_test)

(X_train, Y_train), (X_val, Y_val), (X_test, Y_test) = split_data(scaled_X, scaled_Y)


Total: 101384
Train: 70968 
Validation: 15207 
Test: 15209 


In [145]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def initialize_weights(input_size, hidden_size, output_size):
    np.random.seed(42)  # to reuse the results

    W1 = np.random.randn(input_size, hidden_size) * 0.01 # This is to create a weights matrix with size 2x10 because input_size=2
    # and hidden_size is 10 then 20 weights are generated
    b1 = np.zeros((1, hidden_size)) # 10 Hidden neurons have 10 biases

    W2 = np.random.randn(hidden_size, output_size) * 0.01
    b2 = np.zeros((1, output_size)) # 2 output neurons have 2 biases
    return W1, b1, W2, b2

def forward_propagation(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1 # matrix mx2 (input) multiplied by 2x10 (weight) to form a hidden matrix (mx10) plus 10 biases for 10 neurons
    A1 = sigmoid(Z1)

    Z2 = np.dot(A1, W2) + b2 # matrix mx10 (hidden) multiplied by 10x2 (weight) to create an output matrix of mx2, plus 2 bias for 2 neurons
    A2 = sigmoid(Z2)

    cache = (Z1, A1, Z2, A2)
    return A2, cache

def backward_propagation(X, Y, cache, W1, b1, W2, b2, learning_rate, momentum, 
                         prev_dW1=None, prev_db1=None, prev_dW2=None, prev_db2=None):
    
    Z1, A1, Z2, A2 = cache
    m = X.shape[0]

    # Initialize prev_d if it is the first epoch
    if prev_dW1 is None: prev_dW1 = np.zeros_like(W1) # matrix has the same size as matrix w1 but all values=0
    if prev_db1 is None: prev_db1 = np.zeros_like(b1)
    if prev_dW2 is None: prev_dW2 = np.zeros_like(W2)
    if prev_db2 is None: prev_db2 = np.zeros_like(b2)

    # --- OUTPUT layer (W2) ---
    # η x δk(t) x h(t)= η x (deriv of activate dunction x error) x h(t)
    # y(1-y) * Error
    dA2 = A2 - Y # matrix (mx2) = (y^-y)
    Delta_D2 = dA2 * sigmoid_derivative(A2) # Error * y(1-y)=(y^-y)*y^*(1-y^)=(A2-Y)A2(1-A2)
    # multiply each value in the same position together 
    # this is delta=derivative x error and the dimensions are both (mx2) and (mx2)
    dW2 = (1/m) * np.dot(A1.T, Delta_D2) # multiply by previous layer (hidden)
    #Delta_D2=(mx2)
    #A1_hidden=(mx10)-->A1_hidden.T=(10xm)
    #dot(A1_hidden.T, Delta_D2)=(10x2) = size W2 from hidden to output
    db2 = (1/m) * np.sum(Delta_D2, axis=0, keepdims=True)

    # Hidden Layer (Layer 1)
    # The formular from lecture: η * h(1-h)* sum[(y^-y)*y^(1-y^)*w] *x (lamda=1)
    # the old w to each hidden neuron 
    dA1 = np.dot(Delta_D2, W2.T) #  # Delta_D2 = dA2 * sigmoid_derivative(A2) = sum[(y^-y)*y^(1-y^)*w)= (A2-Y)A2(1-A2)*w
    Delta_D1 = dA1 * sigmoid_derivative(A1)
    dW1 = (1/m) * np.dot(X.T, Delta_D1) 
    db1 = (1/m) * np.sum(Delta_D1, axis=0, keepdims=True)

    # Momentum update
    # Updating w1, w2
    W1_update = learning_rate * dW1 + momentum * prev_dW1
    W2_update = learning_rate * dW2 + momentum * prev_dW2
    
    W1 -= W1_update
    W2 -= W2_update

    # Updating b1, b2
    b1_update = learning_rate * db1 + momentum * prev_db1
    b2_update = learning_rate * db2 + momentum * prev_db2
    
    b1 -= b1_update
    b2 -= b2_update

    return W1, b1, W2, b2, dW1, db1, dW2, db2

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


In [146]:
# FUNCTION to save matrix (W1, W2) as a text file
def save_matrix_manual(filename, data):
    # Open the file in write mode
    with open(filename, 'w') as f:
        # Loop through each row of the matrix
        for row in data:
            # Create an empty string to hold the row data
            line_str = ""
            # Loop through each element in the row
            for i, val in enumerate(row):
                # Convert each element to a string
                line_str += str(val)
                # Add a comma if it's not the last element in the row
                if i < len(row) - 1:
                    line_str += ","
            # Write the row to the file and move to the next line
            f.write(line_str + "\n")

# FUNCTION to save vector (b1, b2) as a text file
def save_vector_manual(filename, data):
    with open(filename, 'w') as f: #overwrite vs append
        # Check if the data is a list or a 1D array
        try:
            # Try to iterate over the data (assuming it's a list of values)
            for val in data:
                f.write(str(val) + "\n")
        except:
            # If it's a single value 
            f.write(str(data) + "\n")
            
def train(X_train, Y_train, X_val, Y_val,
          input_size, hidden_size, output_size,
          epochs, lr, momentum):
    
    # Initialize weights
    W1, b1, W2, b2 = initialize_weights(input_size, hidden_size, output_size)
    
    # Initialize previous gradients for Momentum
    prev_dW1, prev_db1, prev_dW2, prev_db2 = None, None, None, None

    # 1. Initialize for loss logging
    train_losses = []
    val_losses = []
    
    # 3. Track best loss and save the best weights
    best_val_rmse = float('inf')
    best_W1, best_b1, best_W2, best_b2 = W1.copy(), b1.copy(), W2.copy(), b2.copy()

    for epoch in range(epochs):
        # Training (Forward + Backward)
        y_train_pred, cache = forward_propagation(X_train, W1, b1, W2, b2)
        W1, b1, W2, b2, prev_dW1, prev_db1, prev_dW2, prev_db2 = backward_propagation(
            X_train, Y_train, cache, 
            W1, b1, W2, b2, 
            lr, momentum,
            prev_dW1, prev_db1, prev_dW2, prev_db2
        )

        # Validation
        y_val_pred, _ = forward_propagation(X_val, W1, b1, W2, b2)
        train_rmse = rmse(Y_train, y_train_pred)
        val_rmse = rmse(Y_val, y_val_pred)

        # 1. Log loss
        train_losses.append(train_rmse)
        val_losses.append(val_rmse)
        mse_loss = np.mean(np.square(Y_train - y_train_pred)) 
        # 3. Update and save the best weights
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            # Save the best model parameters
            best_W1 = W1.copy()
            best_b1 = b1.copy()
            best_W2 = W2.copy()
            best_b2 = b2.copy()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Train RMSE={train_rmse:.4f}, Val RMSE={val_rmse:.4f} (Best Val: {best_val_rmse:.4f}), Loss: {mse_loss:.8f}")

    # --- Training complete ---
    print("\n--- Training complete. Finalizing parameters ---")
    
    # Save files
    save_matrix_manual('Assignment Code/W1_final.npy', best_W1)
    save_matrix_manual('Assignment Code/W2_final.npy', best_W2)
    save_vector_manual('Assignment Code/b1_final.npy', best_b1)
    save_vector_manual('Assignment Code/b2_final.npy', best_b2)

    # Save loss log manually (text file format)
    with open('loss_logging.txt', 'w') as f:
        f.write("Epoch\tTrain RMSE\tVal RMSE\n")
        for epoch in range(epochs):
            f.write(f"{epoch}\t{train_losses[epoch]:.4f}\t{val_losses[epoch]:.4f}\n")
    
    print("Successfully saved loss log file: loss_logging.npz")
    print("Successfully saved 4 files: W1_final.npy, W2_final.npy, b1_final.npy, b2_final.npy")
    
    # Return the best weights
    return best_W1, best_b1, best_W2, best_b2

In [147]:
best_W1, best_b1, best_W2, best_b2 = train(X_train, Y_train, X_val, Y_val,
                       input_size=2, hidden_size=8, output_size=2,
                       epochs=100, lr=0.01, momentum=0.01)


Epoch 0: Train RMSE=0.1656, Val RMSE=0.1649 (Best Val: 0.1649), Loss: 0.02742655
Epoch 10: Train RMSE=0.1651, Val RMSE=0.1644 (Best Val: 0.1644), Loss: 0.02727238
Epoch 20: Train RMSE=0.1647, Val RMSE=0.1639 (Best Val: 0.1639), Loss: 0.02712153
Epoch 30: Train RMSE=0.1643, Val RMSE=0.1635 (Best Val: 0.1635), Loss: 0.02698163
Epoch 40: Train RMSE=0.1639, Val RMSE=0.1631 (Best Val: 0.1631), Loss: 0.02685190
Epoch 50: Train RMSE=0.1635, Val RMSE=0.1628 (Best Val: 0.1628), Loss: 0.02673160
Epoch 60: Train RMSE=0.1632, Val RMSE=0.1625 (Best Val: 0.1625), Loss: 0.02662004
Epoch 70: Train RMSE=0.1628, Val RMSE=0.1621 (Best Val: 0.1621), Loss: 0.02651661
Epoch 80: Train RMSE=0.1625, Val RMSE=0.1619 (Best Val: 0.1619), Loss: 0.02642071
Epoch 90: Train RMSE=0.1623, Val RMSE=0.1616 (Best Val: 0.1616), Loss: 0.02633179

--- Training complete. Finalizing parameters ---
Successfully saved loss log file: loss_logging.npz
Successfully saved 4 files: W1_final.npy, W2_final.npy, b1_final.npy, b2_final.n

In [148]:
def test(X_test, Y_test, W1, b1, W2, b2):
    
    # 1. Forward Propagation
    y_test_pred, _ = forward_propagation(X_test, W1, b1, W2, b2)
    
    # 2. rmse
    test_rmse = rmse(Y_test, y_test_pred)
    
    print(f"Test RMSE : {test_rmse:.4f}")
    
    return test_rmse, y_test_pred

In [149]:
test_error, predictions = test(
    X_test, Y_test, 
    best_W1, best_b1, best_W2, best_b2
)

Test RMSE : 0.1628
